# 📝 질의 변환 과제 LV2(응용)

교안 02의 질의 변환 다섯 가지를 차례로 적용합니다. 과거 Neo4j 릴리스 노트 5개(`lv2_docs.json`, 영어)를 쓰며, 원문의 옛 버전은 검색 연습용 내용일 뿐 설치할 버전과 관계없습니다. 하이브리드 검색기 `hybrid`는 LV1에서 연습했으므로 `[제공 코드]`로 준비합니다.

- 1~2번: Self-Query로 검색어와 조건을 나누고, 같은 조건을 BM25·Dense에 함께 적용
- 3~6번: HyDE, Multi-Query, Step-back, Decomposition으로 검색하고 변환 전후 Recall 비교
- 7번: 실제 검색한 원문으로 답하고 기록 저장
- 8번: 상황에 맞는 변환 고르기와 결과 해석(서술)

`.env`의 OpenAI 키가 필요합니다. GPT 요청은 모두 6회(1·3·4·5·6·7번에 한 번씩)이고, 임베딩은 문서 5개와 검색 질문마다 요청합니다(자가채점의 확인 검색 포함). Recall 비교는 교안의 `compare_recall`을 `[제공 코드]`로 주므로 직접 계산하지 않습니다. 변환이 늘 Recall을 올리지는 않으니, 오르지 않은 경우도 출력으로 원인을 살펴보세요.

**풀이 방법**: 준비 셀과 `[제공 코드]` 셀을 위에서부터 실행한 뒤, 문항 순서대로 답안 셀을 채우고 바로 아래 `# [자가채점]` 셀로 확인하세요. 앞 문항의 변수를 다음 문항에서 이어 씁니다. 검색 순위와 모델이 만든 문장은 고정 정답이 아니므로 자가채점은 처리 순서와 원문 대응만 확인합니다. 결과가 맞는지는 출력된 원문을 읽어 판단하세요. 자가채점이 검색을 다시 해 대조하는 문항은 순위가 거의 같은 문서 때문에 드물게 실패할 수 있으니, 그때는 답안 셀부터 다시 실행하세요.


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하고 준비 셀을 위에서부터 실행하세요. 경로(`material_dir`·`data_dir`·`output_dir`)와 `read_json`·`save_json`은 앞 단원과 같습니다. 첫 셀에서 `langchain-community` 유지보수 종료를 알리는 경고가 한 번 보일 수 있습니다. `BM25Retriever`를 이 패키지에서 가져오기 때문이며 실행에는 문제가 없습니다.


In [ ]:
# 문서·검색기·모델에 필요한 라이브러리를 가져옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


임베딩은 `text-embedding-3-large`의 768차원입니다. `check_embedding_ctx_length=False`는 자동 길이 검사·분할을 끕니다. API 키는 `.env`에서 읽습니다.


In [ ]:
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 이 셀은 모델 설정만 준비합니다. GPT 요청은 뒤의 체인 invoke에서 발생합니다.
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 적재·검색 셀에서 요청합니다.")


절 하나가 이미 짧은 발췌라서 다시 나누지 않고 레코드 하나를 검색 단위 하나로 씁니다. 이 단위를 고정한 채 검색 방식과 질문을 바꿔 결과를 비교합니다. `make_documents`는 원문 ID·제목·출처에 필터용 메타데이터를 더해 `Document`를 만듭니다.


In [ ]:
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 사용하고 출처·필터 메타데이터를 담습니다.
    return [Document(
        id=record["doc_id"],
        page_content=record["text"],
        metadata={"source_id": record["doc_id"], "title": record["title"],
                  "url": record["url"], "source": record["source"], **record["metadata"]},
    ) for record in records]


In [ ]:
# 전체 목록을 먼저 보고 질문에 필요한 필터 필드를 확인합니다.
records = read_json("lv2_docs.json")
display(pd.DataFrame(records)[["doc_id", "title", "metadata"]])
print(records[0]["text"])
documents = make_documents(records)


In [ ]:
def show_results(documents):
    """앞 5개 결과의 원문 ID·메타데이터·본문을 모든 열과 함께 보여 줍니다."""
    # 빈 결과는 필터를 바꾸지 않고 그대로 알립니다.
    if not documents:
        print("조건에 맞는 검색 결과가 없습니다.")
        return
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    # 열 순서를 고정해야 표를 나란히 비교할 수 있습니다. url은 길어서 뺍니다.
    rows = [{"display_order": order, "source_id": doc.metadata["source_id"], "title": doc.metadata["title"],
             **{key: doc.metadata[key] for key in sorted(doc.metadata) if key not in {"source_id", "title", "url"}},
             "text": doc.page_content} for order, doc in enumerate(documents, start=1)]
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head())


아래 `[제공 코드]` 셀은 실행만 하세요. 교안 02와 같은 질의 변환 도구, 벡터 저장소, 하이브리드 검색기, Self-Query 추출 규칙 `query_schema_prompt`(교안 02 따라하기와 같은 규칙), 조건 검색 함수 `search_with_filter`, 전후 비교 함수 `compare_recall`, 중복 제거 함수 `unique_documents`를 준비합니다.


In [ ]:
# [제공 코드]
# 질의 생성과 메타데이터 조건 번역에 쓰는 도구를 가져옵니다.
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
from pydantic import BaseModel, Field


In [ ]:
# [제공 코드]
# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF 표기 통일: 재택･원격근무 → 재택·원격근무
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    return [token.form.lower() for token in kiwi.tokenize(text)
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


In [ ]:
# [제공 코드]
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day47_lv2_releases", embedding_function=embedding_model)
vector_store.reset_collection()
# make_documents가 넣은 Document.id가 저장소 ID가 되므로 원문 ID가 검색 결과까지 그대로 이어집니다.
added_ids = vector_store.add_documents(documents)
print("day47_lv2_releases 적재 수:", len(added_ids))


In [ ]:
# [제공 코드]
# LV1에서 연습한 하이브리드 검색기입니다. 문서가 5개라 검색기마다 TOP_K=2개를 가져옵니다.
TOP_K = 2
bm25 = BM25Retriever.from_documents(documents, preprocess_func=kiwi_tokenize, k=TOP_K)
dense = vector_store.as_retriever(search_kwargs={"k": TOP_K})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="source_id")


In [ ]:
# [제공 코드]
# 검색어와 조건을 추출하는 규칙입니다.
query_schema_prompt = PromptTemplate.from_template(
    "마지막 User Query를 검색어와 메타데이터 조건으로 나누세요. 예시의 값을 복사하지 마세요. "
    "JSON 객체 하나만 출력하세요. query에는 본문에서 찾을 핵심 주제를, filter에는 명시된 조건을 넣습니다. "
    "주제가 있으면 query를 비우지 말고, 조건이 없을 때만 filter를 NO_FILTER로 적으세요. "
    "filter는 비교 연산자({allowed_comparators})와 논리 연산자({allowed_operators})로 표현합니다. "
    "Data Source에 정의된 필드만 사용하며, 숫자는 문자열로 바꾸지 마세요."
)


In [ ]:
# [제공 코드]
def search_with_filter(query, where, bm25, vector_store, hybrid, k):
    """기존 BM25를 재사용하고 같은 조건의 Dense 결과와 합칩니다."""
    # 두 검색에 적용할 조건 일치 문서 ID를 Chroma에서 조회합니다.
    matched = vector_store.get(where=where, include=["metadatas"])
    allowed_ids = {item["source_id"] for item in matched["metadatas"]}
    if not allowed_ids:
        return []
    if not query.strip():
        candidates = [doc for doc in bm25.docs if doc.metadata["source_id"] in allowed_ids]
        return sorted(candidates, key=lambda doc: doc.metadata["source_id"])[:k]

    # BM25 점수를 계산합니다.
    scores = bm25.vectorizer.get_scores(bm25.preprocess_func(query))

    # 조건에 맞는 문서와 점수만 남깁니다.
    scored_candidates = [
        (doc, score) for doc, score in zip(bm25.docs, scores)
        if doc.metadata["source_id"] in allowed_ids
    ]

    # 조건에 맞는 문서를 BM25 점수 내림차순으로 정렬해 상위 k개를 선택합니다.
    scored_candidates.sort(key=lambda item: item[1], reverse=True)
    bm25_results = [doc for doc, score in scored_candidates[:k]]

    # Dense도 같은 조건을 적용해 실제 벡터 DB에서 검색합니다.
    dense_results = vector_store.similarity_search(query, k=k, filter=where)

    # 문서별 가중 RRF 점수를 합산해 내림차순으로 정렬하고 상위 k개를 반환합니다.
    return hybrid.weighted_reciprocal_rank([bm25_results, dense_results])[:k]


In [ ]:
# [제공 코드]
def compare_recall(before, after, expected_ids):
    """변환 전후의 전체 후보 Recall과 중복을 제외한 문서 수를 비교합니다."""
    expected_ids = set(expected_ids)
    rows = []
    for stage, results in [("변환 전", before), ("변환 후", after)]:
        retrieved_ids = {doc.metadata["source_id"] for doc in results}
        rows.append({
            "stage": stage,
            "recall": len(retrieved_ids & expected_ids) / len(expected_ids),
            "retrieved_count": len(retrieved_ids),
            "expected_count": len(expected_ids),
            "missing_ids": sorted(expected_ids - retrieved_ids),
        })
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head().round(3))


In [ ]:
# [제공 코드]
# 여러 질문이 같은 원문을 찾아도 답변 문맥에는 한 번만 넣습니다.
def unique_documents(documents):
    """원문 ID를 기준으로 첫 등장 순서를 유지하며 중복을 제거합니다."""
    by_id = {}
    for doc in documents:
        source_id = doc.metadata["source_id"]
        if source_id not in by_id:
            by_id[source_id] = doc
    return list(by_id.values())


## 1. Self-Query로 검색어와 조건을 나눕니다

**배경**: ‘stable 릴리스에서’ 같은 조건은 본문 검색어가 아니라 메타데이터 조건입니다. 교안 02 1절처럼 조건으로 쓸 필드를 알려 주고, 질문을 검색어와 조건으로 나눈 뒤 그 조건으로 벡터 검색합니다.

**요구사항**:

- **`metadata_field_info`**: `AttributeInfo` 두 개를 담은 목록입니다. 준비 셀의 표를 보고 `series`(버전 계열, 값 3.3·3.2·3.0·2.3·2.2)와 `stage`(배포 단계, 값 alpha·beta·rc·stable)를 이 순서로 정의하세요. 둘 다 `type="string"`이고, `description`에 값 목록을 적으세요.
- **`self_query`**: 교안 02 1절과 같은 방법으로 `SelfQueryRetriever.from_llm`으로 만드세요. 모델은 `llm`, 저장소는 `vector_store`, 문서 설명은 `"Historical Neo4j release notes"`, 필드 정보는 `metadata_field_info`, 추출 규칙은 제공된 `query_schema_prompt`(`chain_kwargs`의 `"schema_prompt"`로), 번역기는 `ChromaTranslator()`, 검색 개수는 `search_kwargs={"k": TOP_K}`로 줍니다.
- **`question`**: 문자열 `"Find data import changes in releases whose stage is stable."`를 저장하세요. 2번에서도 씁니다.
- **`parsed`**: `self_query.query_constructor`에 `{"query": question}`을 넣어 **한 번** 추출한 결과입니다(2번에서 같은 조건을 BM25에도 넘기려고 `self_query.invoke` 대신 조건만 꺼냅니다).
- **`query_text`**, **`search_kwargs`**, **`where`**: `ChromaTranslator().visit_structured_query(parsed)`가 돌려주는 두 값과, `search_kwargs`의 `"filter"` 값(`get` 사용, 조건이 없으면 `None`)입니다.
- **`self_query_results`**: `vector_store.similarity_search`에 `query_text`, `k=TOP_K`, `filter=where`를 넣은 결과입니다.

**예시**: 질문대로 추출되면 `where`는 `{'stage': {'$eq': 'stable'}}`이고 결과는 stable 릴리스 2개입니다. 모델이 조건을 다르게 뽑아도 고치지 말고 그대로 쓰세요. 추출 결과는 실행마다 달라질 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 필드를 먼저 정의하고, 조건 추출(모델 요청)은 한 번만 한 뒤 번역한 조건을 검색에 그대로 넘깁니다.

세부구현:
1. AttributeInfo로 series와 stage 필드를 정의합니다.
2. 교안 1절의 from_llm 호출을 참고해 self_query를 만듭니다.
3. query_constructor의 invoke로 parsed를 받고, ChromaTranslator로 번역해 filter를 where로 꺼냅니다.
4. similarity_search로 검색하고 show_results로 stage를 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert isinstance(metadata_field_info, list) and all(isinstance(field, AttributeInfo) for field in metadata_field_info), 'metadata_field_info는 AttributeInfo 객체를 담은 목록(list)이어야 합니다. 딕셔너리나 튜플로 담지 마세요.'
assert [(field.name, field.type) for field in metadata_field_info] == [('series', 'string'), ('stage', 'string')], 'metadata_field_info에는 series, stage 순서로 type="string"인 AttributeInfo를 담으세요.'
assert all(value in metadata_field_info[1].description for value in ['alpha', 'beta', 'rc', 'stable']), 'stage의 description에 배포 단계 값(alpha, beta, rc, stable)을 적으세요.'
assert isinstance(self_query, SelfQueryRetriever) and self_query.vectorstore is vector_store, 'self_query는 vector_store로 만든 SelfQueryRetriever입니다.'
assert self_query.search_kwargs == {'k': TOP_K}, 'search_kwargs={"k": TOP_K}를 지정하세요.'
assert isinstance(self_query.structured_query_translator, ChromaTranslator), 'structured_query_translator=ChromaTranslator()를 지정하세요.'
assert question == 'Find data import changes in releases whose stage is stable.', '지문의 질문 문장을 그대로 question에 저장하세요.'
assert (query_text, search_kwargs) == ChromaTranslator().visit_structured_query(parsed), 'parsed를 ChromaTranslator로 번역한 두 값을 query_text, search_kwargs에 담으세요.'
assert where == search_kwargs.get('filter'), 'where에는 search_kwargs의 filter 값을 담으세요. 조건이 없으면 None입니다.'
allowed_ids = {item['source_id'] for item in vector_store.get(where=where, include=['metadatas'])['metadatas']}
assert isinstance(self_query_results, list) and len(self_query_results) == min(TOP_K, len(allowed_ids)), 'similarity_search에 k=TOP_K와 filter=where를 넣은 결과를 self_query_results에 담으세요.'
assert {doc.metadata['source_id'] for doc in self_query_results} <= allowed_ids, '추출한 where를 고치지 말고 similarity_search의 filter에 그대로 넣으세요.'
print("✅ 1번 통과!")


## 2. 같은 조건을 BM25와 Dense에 함께 적용합니다

**배경**: 조건을 한쪽 검색에만 걸면 다른 쪽에서 조건 밖 문서가 섞입니다. 제공된 `search_with_filter`로 같은 조건을 두 검색에 적용하고, 조건 없이 검색한 결과와 비교합니다.

**요구사항**:

- **`expected_ids`**: 이 질문의 정답 원문 ID 집합 `{"release3", "release4"}`입니다(stable 릴리스의 LOAD CSV·import 변경).
- **`filtered_results`**: `search_with_filter`에 1번의 `query_text`, `where`와 제공된 `bm25`, `vector_store`, `hybrid`, `k=TOP_K`를 이 순서로 넣은 결과입니다.
- **`before_results`**: 조건 없이 `hybrid`로 1번 `question`을 검색한 결과의 앞 `TOP_K`개입니다.
- **출력과 비교**: `show_results`로 `before_results`와 `filtered_results`를 차례로 출력해 `stage`를 비교하고, 제공된 `compare_recall`에 `before_results`, `filtered_results`, `expected_ids`를 이 순서로 넣어 표를 출력하세요.

**예시**: `filtered_results`의 문서는 모두 `where` 조건을 만족합니다. 조건 없이 검색한 `before_results`에는 stable이 아닌 릴리스(예: rc)가 섞일 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조건은 새로 뽑지 않고 1번 결과를 그대로 재사용합니다.

세부구현:
1. expected_ids를 지문대로 저장합니다.
2. search_with_filter에 인자를 순서대로 넣어 filtered_results를 받습니다.
3. hybrid.invoke 결과를 TOP_K개로 잘라 before_results에 담습니다.
4. 두 결과를 출력하고 compare_recall로 전후를 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert expected_ids == {'release3', 'release4'}, 'expected_ids는 {"release3", "release4"}입니다.'
allowed_ids = {item['source_id'] for item in vector_store.get(where=where, include=['metadatas'])['metadatas']}
assert isinstance(filtered_results, list) and {doc.metadata['source_id'] for doc in filtered_results} <= allowed_ids, '1번의 where를 그대로 search_with_filter에 넣으세요. 결과는 모두 조건을 만족해야 합니다.'
assert [doc.metadata['source_id'] for doc in filtered_results] == [doc.metadata['source_id'] for doc in search_with_filter(query_text, where, bm25, vector_store, hybrid, k=TOP_K)], 'search_with_filter에는 1번의 query_text와 where, 제공된 bm25·vector_store·hybrid, k=TOP_K를 넣으세요. 원질문이 아니라 추출한 검색어를 씁니다.'
assert [doc.metadata['source_id'] for doc in before_results] == [doc.metadata['source_id'] for doc in hybrid.invoke(question)][:TOP_K], 'before_results에는 조건 없이 hybrid로 1번 question을 검색한 결과의 앞 TOP_K개를 담으세요.'
print("✅ 2번 통과!")


아래 `[제공 코드]`는 교안 02의 HyDE 체인 `hyde_chain`입니다. 질문을 받아 검색에 쓸 가상의 문단을 만듭니다. 문서가 영어라 가상문단도 질문과 같은 언어로 쓰도록 출력 언어만 바꿨습니다.


In [ ]:
# [제공 코드]
# 문서가 영어라 검색 입력도 영어여야 BM25·Dense가 맞춰 봅니다. 교안 프롬프트에서 출력 언어 규칙만 바꿨습니다.
# 가상문서는 검색에만 쓰고 최종 답변은 실제 원문으로 작성합니다.
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "질문에 답할 내용이 담긴 가상의 원문을 3문장 이내로 작성하세요. "
               "각 요구에 필요한 핵심 개념을 해당 분야의 표준 용어로 명시하고 설명하세요. "
               "질문을 다시 쓰거나 관련 주제를 추가하지 마세요. "
               "도서 검색 질문에는 해당 내용을 가르치는 책의 소개문을 쓰되 책 제목·저자·출판사는 만들지 마세요. "
               "확인되지 않은 구체적인 수치나 규정을 지어내지 마세요. "
               "요청문·검색어 목록 없이 질문과 같은 언어의 원문만 출력하세요."),
    ("human", "{question}"),
])
hyde_chain = hyde_prompt | llm | StrOutputParser()


## 3. HyDE의 가상문단으로 Dense를 검색합니다

**배경**: 짧은 질문은 문서의 설명 문장과 표현이 다를 수 있습니다. HyDE는 가상의 답 문단을 만들어 그 문단으로 Dense를 검색하고, BM25에는 원질문을 넣은 뒤 두 결과를 RRF로 합칩니다.

**요구사항**:

- **`hyde_question`**: 문자열 `"How were schema and indexes improved?"`를 저장하세요.
- **`hyde_expected_ids`**: 정답 원문 ID 집합 `{"release0", "release1"}`입니다(인덱스 기본값·수동 인덱스, 인덱스 로깅·제약 조건 변경).
- **`hypothesis`**: `hyde_chain`에 `{"question": hyde_question}`을 넣어 만든 가상문단 문자열입니다. 출력하세요.
- **`hyde_bm25`**, **`hyde_dense`**, **`question_dense`**: `bm25`로 **원질문**을, `dense`로 **가상문단**을, 비교용으로 `dense`로 **원질문**을 검색한 결과입니다. `hyde_dense`와 `question_dense`의 원문 ID를 나란히 출력하세요.
- **`hyde_results`**: `hybrid.weighted_reciprocal_rank`에 `[hyde_bm25, hyde_dense]`를 이 순서로 넣은 결과의 앞 `TOP_K`개입니다.
- **`hyde_before`**: 원질문을 `hybrid`로 검색한 결과의 앞 `TOP_K`개입니다. `compare_recall`로 `hyde_before`와 `hyde_results`를 비교하세요.

**예시**: `hyde_results`는 `TOP_K`개입니다. 이 자료는 문서가 5개뿐이라 원질문의 Dense 결과가 이미 가상문단의 결과와 비슷해 Recall이 그대로일 수 있습니다. `hyde_dense`와 `question_dense`를 비교해 가상문단이 Dense 순위를 바꿨는지 확인하세요. 가상문단은 검색 입력일 뿐 사실이 아니며 실행마다 달라집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 가상문단은 Dense에만, 원질문은 BM25에 넣고, 두 결과를 hybrid의 RRF로 합칩니다.

세부구현:
1. hyde_chain으로 가상문단을 만듭니다.
2. bm25에는 원질문, dense에는 가상문단을 넣고, 비교용으로 dense에 원질문도 넣습니다.
3. weighted_reciprocal_rank에 BM25 결과, 가상문단 Dense 결과 순서로 넣고 앞 TOP_K개를 자릅니다.
4. 원질문의 하이브리드 결과와 compare_recall로 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert hyde_question == 'How were schema and indexes improved?', '지문의 질문 문장을 그대로 hyde_question에 저장하세요.'
assert hyde_expected_ids == {'release0', 'release1'}, 'hyde_expected_ids는 {"release0", "release1"}입니다.'
assert isinstance(hypothesis, str) and hypothesis.strip() and hypothesis != hyde_question, 'hypothesis에는 hyde_chain이 만든 가상문단 문자열을 담으세요.'
assert [doc.metadata['source_id'] for doc in hyde_bm25] == [doc.metadata['source_id'] for doc in bm25.invoke(hyde_question)], 'hyde_bm25에는 bm25로 원질문을 검색한 결과를 담으세요. 가상문단은 Dense에만 넣습니다.'
assert [doc.metadata['source_id'] for doc in hyde_dense] == [doc.metadata['source_id'] for doc in dense.invoke(hypothesis)], 'hyde_dense에는 dense로 가상문단(hypothesis)을 검색한 결과를 담으세요.'
assert [doc.metadata['source_id'] for doc in question_dense] == [doc.metadata['source_id'] for doc in dense.invoke(hyde_question)], 'question_dense에는 비교용으로 dense에 원질문을 넣은 결과를 담으세요.'
assert len(hyde_results) == TOP_K and [doc.metadata['source_id'] for doc in hyde_results] == [doc.metadata['source_id'] for doc in hybrid.weighted_reciprocal_rank([hyde_bm25, hyde_dense])][:TOP_K], 'hyde_results에는 hybrid의 RRF로 hyde_bm25, hyde_dense를 이 순서로 합친 결과의 앞 TOP_K개를 담으세요.'
assert [doc.metadata['source_id'] for doc in hyde_before] == [doc.metadata['source_id'] for doc in hybrid.invoke(hyde_question)][:TOP_K], 'hyde_before에는 hybrid로 원질문을 검색한 결과의 앞 TOP_K개를 담으세요.'
print("✅ 3번 통과!")


아래 `[제공 코드]`는 교안 02의 Multi-Query 프롬프트 `multi_prompt`와, 생성 질문을 출력하는 `show_queries`입니다.


In [ ]:
# [제공 코드]
# from_llm의 기본 줄 단위 파서가 읽을 수 있게 번호 없이 한 줄에 하나씩 받습니다.
multi_prompt = ChatPromptTemplate.from_template(
    "질문과 같은 의미를 유지하는 검색 질문을 서로 다른 표현으로 정확히 3개 작성하세요. "
    "일상 표현을 관련 분야의 표준 용어로 바꾼 질문도 포함하세요. "
    "원문의 정보 요구·고유명사·조건을 유지하세요. 질문의 언어를 유지하세요. "
    "설명·번호·빈 줄 없이 한 줄에 질문 하나만 출력하세요.\n질문: {question}"
)


In [ ]:
# [제공 코드]
def show_queries(queries):
    """실제로 검색에 사용할 생성 질문을 출력하고 그대로 전달합니다."""
    # 출력용으로 LLM을 다시 호출하지 않고 기존 체인의 결과를 관찰합니다.
    print("생성 질문:", queries)
    return queries


## 4. Multi-Query로 같은 뜻의 질문을 늘립니다

**배경**: 질문과 문서의 표현이 다르면 한 가지 표현으로는 근거를 놓칠 수 있습니다. Multi-Query는 같은 뜻의 질문을 여러 개 만들어 각각 하이브리드 검색하고 결과를 합칩니다.

**요구사항**:

- **`multi_question`**: 문자열 `"How did Neo4j change the way it keeps a history of events?"`를 저장하세요. 문서는 ‘history of events’ 대신 ‘logging’·‘log’라고 씁니다.
- **`multi_expected_ids`**: 정답 원문 ID 집합 `{"release1", "release2", "release4"}`입니다(로그 관련 변경이 있는 세 릴리스).
- **`multi_query`**: `MultiQueryRetriever.from_llm`에 `retriever=hybrid`, `llm=llm`, `prompt=multi_prompt`, `include_original=True`를 넣어 만듭니다. 만든 뒤 `multi_query.llm_chain`을 `multi_query.llm_chain | show_queries`로 바꿔 생성 질문이 출력되게 하세요.
- **`multi_query_chain`**: `multi_query | unique_documents`로 검색과 원문 ID 중복 제거를 연결한 체인입니다.
- **`multi_before`**, **`multi_results`**: 원질문을 `hybrid`로 검색한 **자르지 않은 전체 결과**와, `multi_query_chain`으로 검색한 결과입니다. `multi_results`가 여러 질문의 합집합이라 비교 기준도 교안처럼 자르지 않습니다. `compare_recall`로 비교하세요.

**예시**: `include_original=True`라서 원질문의 결과가 `multi_results`에 모두 들어갑니다. 생성 질문이 ‘logging’ 같은 문서의 표현을 쓰면 원질문이 놓친 릴리스를 더 찾습니다. 합친 목록의 순서는 관련성 순위가 아닙니다.

<details><summary>힌트</summary>

```text
접근방법:
- 원질문도 함께 검색하도록 만들고, 결과는 원문 ID로 중복을 없앱니다.

세부구현:
1. MultiQueryRetriever.from_llm으로 multi_query를 만들고 llm_chain 뒤에 show_queries를 연결합니다.
2. | 연산자로 multi_query와 unique_documents를 이어 multi_query_chain을 만듭니다.
3. 원질문의 hybrid 전체 결과와 multi_query_chain 결과를 각각 받습니다.
4. compare_recall로 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
from langchain_core.runnables import RunnableLambda, RunnableSequence
assert multi_question == 'How did Neo4j change the way it keeps a history of events?', '지문의 질문 문장을 그대로 multi_question에 저장하세요.'
assert multi_expected_ids == {'release1', 'release2', 'release4'}, 'multi_expected_ids는 {"release1", "release2", "release4"}입니다.'
assert isinstance(multi_query, MultiQueryRetriever) and multi_query.retriever is hybrid, 'multi_query는 retriever=hybrid로 만든 MultiQueryRetriever입니다.'
assert multi_query.include_original is True, 'include_original=True로 원질문도 함께 검색하세요.'
assert isinstance(multi_query.llm_chain.last, RunnableLambda) and multi_query.llm_chain.last.func is show_queries, 'multi_query.llm_chain 뒤에 show_queries를 | 로 연결하세요.'
assert isinstance(multi_query_chain, RunnableSequence) and isinstance(multi_query_chain.last, RunnableLambda) and multi_query_chain.last.func is unique_documents, 'multi_query_chain은 multi_query 뒤에 unique_documents를 | 로 연결한 체인입니다.'
assert [doc.metadata['source_id'] for doc in multi_before] == [doc.metadata['source_id'] for doc in hybrid.invoke(multi_question)], 'multi_before에는 hybrid로 원질문을 검색한 결과를 자르지 않고 담으세요.'
multi_ids = [doc.metadata['source_id'] for doc in multi_results]
assert isinstance(multi_results, list) and len(multi_ids) == len(set(multi_ids)), 'multi_results는 multi_query_chain으로 검색한 결과입니다.'
assert {doc.metadata['source_id'] for doc in multi_before} <= set(multi_ids), 'include_original=True라면 원질문의 hybrid 결과가 multi_results에 모두 들어 있어야 합니다.'
print("✅ 4번 통과!")


아래 `[제공 코드]`는 교안 02의 Step-back 체인 `stepback_chain`입니다. 구체적인 질문에서 배경 개념을 묻는 질문을 만듭니다. 배경 질문도 원질문과 같은 언어로 만들도록 출력 언어 규칙과 예시 한 쌍만 영어로 바꿨습니다.


In [ ]:
# [제공 코드]
# 문서가 영어라 배경 질문도 영어여야 BM25·Dense가 맞춰 봅니다. 교안 프롬프트에서 출력 언어 규칙과 예시 한 쌍만 영어로 바꿨습니다.
# 구체적 사례를 배경 원리로 바꾸는 예시를 함께 보여 줍니다.
stepback_prompt = ChatPromptTemplate.from_messages([
    ("system", "원질문의 구체적인 대상과 작업을 제거하고, 그 배경이 되는 기초 개념 자체를 묻는 질문 하나를 작성하세요. "
               "핵심 개념의 이름을 명시하세요. "
               "원래 대상의 처리 방법을 다른 말로 다시 묻지 마세요. "
               "도서 추천 요청이나 답변 없이 원질문과 같은 언어의 질문만 출력하세요."),
    ("human", "How do I find a car's speed at one exact moment from a log of distance over time?"),
    ("ai", "What is a derivative, and how does it describe an instantaneous rate of change?"),
    ("human", "{question}"),
])
stepback_chain = stepback_prompt | llm | StrOutputParser()


## 5. Step-back으로 배경 질문도 함께 검색합니다

**배경**: ‘왜 그런 일이 생기는가’를 이해하려면 구체적인 변경 내역과 함께 배경 개념이 필요합니다. 원질문과 배경 질문을 각각 하이브리드 검색해 결과를 합칩니다.

**요구사항**:

- **`stepback_question`**: 문자열 `"Why could a cluster briefly have no master, and which release shortened that time?"`를 저장하세요.
- **`stepback_expected_ids`**: 정답 원문 ID 집합 `{"release3"}`입니다(마스터 선출 전환을 빠르게 한 릴리스).
- **`background_question`**: `stepback_chain`에 `{"question": stepback_question}`을 넣어 만든 배경 질문 문자열입니다. 출력하세요.
- **`direct_results`**, **`background_results`**: `hybrid.batch`에 `[stepback_question, background_question]`을 이 순서로 넣어 받은 두 결과를 각각 앞 `TOP_K`개로 자른 목록입니다.
- **`added_ids`**: 배경 질문 결과에만 있는 원문 ID 집합입니다(배경 결과 ID 집합에서 원질문 결과 ID 집합을 뺀 것).
- **`stepback_results`**: `direct_results` 뒤에 `background_results`를 이어 붙여 `unique_documents`로 정리한 목록입니다. `compare_recall`로 `direct_results`와 비교하세요.

**예시**: `stepback_results`는 원질문 결과가 앞에 오고 2~4개입니다. 이 자료에는 일반 원리를 설명하는 문서가 없어, 배경 질문이 새로 찾은 원문(`added_ids`)이 질문과 관련 없을 수 있습니다. 추가된 원문을 읽어 답변에 쓸 근거인지 판단하세요. 배경 질문이 질문이 아니라 답변 문장으로 나오면 셀을 다시 실행하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 배경 질문을 만든 뒤 두 질문을 한 번의 batch로 검색하고, 원질문 결과를 앞에 두고 합칩니다.

세부구현:
1. stepback_chain으로 배경 질문을 만듭니다.
2. hybrid.batch에 두 질문을 목록으로 넣고, 결과 두 개를 각각 앞 TOP_K개로 자릅니다.
3. 두 결과의 source_id 집합 차이로 added_ids를 구합니다.
4. 원질문 결과 + 배경 결과를 unique_documents에 넣고 compare_recall로 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert stepback_question == 'Why could a cluster briefly have no master, and which release shortened that time?', '지문의 질문 문장을 그대로 stepback_question에 저장하세요.'
assert stepback_expected_ids == {'release3'}, 'stepback_expected_ids는 {"release3"}입니다.'
assert isinstance(background_question, str) and background_question.strip() and background_question != stepback_question, 'background_question에는 stepback_chain이 만든 배경 질문 문자열을 담으세요.'
assert isinstance(direct_results, list) and isinstance(background_results, list) and len(direct_results) == TOP_K and len(background_results) == TOP_K, 'hybrid.batch 결과 두 개를 각각 앞 TOP_K개로 잘라 direct_results, background_results에 담으세요.'
assert [doc.metadata['source_id'] for doc in direct_results] == [doc.metadata['source_id'] for doc in hybrid.invoke(stepback_question)][:TOP_K], 'direct_results는 원질문의 결과입니다. hybrid.batch에 원질문, 배경 질문 순서로 넣었는지 확인하세요.'
direct_ids = {doc.metadata['source_id'] for doc in direct_results}
background_ids = {doc.metadata['source_id'] for doc in background_results}
assert isinstance(added_ids, set) and added_ids <= background_ids and not (added_ids & direct_ids) and len(added_ids) == len(background_ids - direct_ids), 'added_ids에는 배경 결과에만 있는 원문 ID를 집합으로 담으세요.'
stepback_ids = [doc.metadata['source_id'] for doc in stepback_results]
assert stepback_ids[:len(direct_results)] == [doc.metadata['source_id'] for doc in direct_results] and len(stepback_ids) == len(set(stepback_ids)) and set(stepback_ids) == direct_ids | background_ids, 'stepback_results는 direct_results 뒤에 background_results를 붙여 unique_documents로 정리한 목록입니다.'
print("✅ 5번 통과!")


아래 `[제공 코드]`는 교안 02의 질문 분해 체인 `decompose_chain`입니다. 결과의 `questions`에 하위 질문 2~3개가 목록으로 담깁니다.


In [ ]:
# [제공 코드]
class SubQuestions(BaseModel):
    """원질문을 나누어 검색할 하위 질문 목록입니다."""
    questions: list[str] = Field(
        description=(
            "원질문의 서로 다른 정보 요구를 하나씩 묻는 질문 2~3개. "
            "각 질문만으로 검색할 수 있게 대상과 조건을 포함하고, "
            "원질문에 없는 요구를 추가하지 않으며 같은 언어로 작성."
        ),
        min_length=2, max_length=3,
    )


# 스키마는 목록 모양을 제어합니다. 질문이 원의도를 보존했는지는 사람이 읽습니다.
decompose_prompt = ChatPromptTemplate.from_messages([
    ("system", "원질문의 서로 다른 정보 요구를 하나씩 묻는 질문 2~3개로 나누세요. "
               "각 질문만으로 검색할 수 있게 대상과 조건을 포함하세요. 원질문에 없는 요구를 추가하지 말고 같은 언어로 작성하세요."),
    ("human", "{question}"),
])
decompose_chain = decompose_prompt | llm.with_structured_output(
    SubQuestions, method="json_schema",
)


## 6. 복합 질문을 하위 질문으로 나눠 검색합니다

**배경**: 한 질문에 서로 다른 요구가 둘 이상 있으면 한 번의 검색이 한쪽 요구에 치우칩니다. 하위 질문으로 나눠 각각 검색하고 결과를 모읍니다.

**요구사항**:

- **`decomposition_question`**: 문자열 `"Which release let the import tool ignore empty strings, and which release sped up master election in a cluster?"`를 저장하세요. 7번에서도 씁니다.
- **`decomposition_expected_ids`**: 정답 원문 ID 집합 `{"release4", "release3"}`입니다(`neo4j-import`의 빈 문자열 옵션, 마스터 선출 전환 단축).
- **`subquestions`**: `decompose_chain`에 `{"question": decomposition_question}`을 넣은 결과입니다. `subquestions.questions`를 출력하세요.
- **`subquestion_results`**: `hybrid.batch`에 `subquestions.questions`를 넣어 받은 결과 목록입니다(하위 질문마다 목록 하나).
- **`all_results`**: `subquestion_results`의 각 목록에서 앞 `TOP_K`개씩 꺼내 순서대로 모은 `Document` 목록입니다.
- **`decomposition_results`**: `all_results`를 `unique_documents`로 정리한 목록입니다. 원질문을 `hybrid`로 검색한 앞 `TOP_K`개 **`decomposition_before`**와 `compare_recall`로 비교하세요.

**예시**: 하위 질문은 2~3개이고 실행마다 문장이 달라질 수 있습니다. 원질문 한 번의 검색은 한쪽 요구에 치우쳐 다른 요구의 근거를 놓치기 쉽고, 하위 질문으로 나누면 두 요구의 근거를 각각 찾을 수 있습니다. 하위 질문이 두 요구를 빠짐없이 다루는지 직접 읽으세요.

<details><summary>힌트</summary>

```text
접근방법:
- 하위 질문을 한 번의 batch로 검색하고, 질문마다 앞 TOP_K개를 모은 뒤 중복을 없앱니다.

세부구현:
1. decompose_chain으로 하위 질문 객체를 받습니다.
2. hybrid.batch에 subquestions.questions를 넣습니다.
3. 결과 목록마다 앞 TOP_K개를 꺼내 순서대로 all_results에 모읍니다.
4. unique_documents로 정리하고 원질문 결과와 compare_recall로 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert decomposition_question == 'Which release let the import tool ignore empty strings, and which release sped up master election in a cluster?', '지문의 질문 문장을 그대로 decomposition_question에 저장하세요.'
assert decomposition_expected_ids == {'release4', 'release3'}, 'decomposition_expected_ids는 {"release4", "release3"}입니다.'
assert isinstance(subquestions.questions, list) and 2 <= len(subquestions.questions) <= 3, 'subquestions에는 decompose_chain의 결과를 담으세요. questions에 하위 질문 2~3개가 있어야 합니다.'
assert isinstance(subquestion_results, list) and len(subquestion_results) == len(subquestions.questions) and all(len(results) >= TOP_K for results in subquestion_results), 'subquestion_results에는 hybrid.batch에 subquestions.questions를 넣은 결과를 담으세요(하위 질문마다 목록 하나).'
assert len(all_results) == TOP_K * len(subquestion_results) and all(all_results[i * TOP_K:(i + 1) * TOP_K] == results[:TOP_K] for i, results in enumerate(subquestion_results)), 'all_results에는 각 결과 목록에서 앞 TOP_K개씩을 먼저 자른 뒤 하위 질문 순서대로 모으세요.'
decomposition_ids = [doc.metadata['source_id'] for doc in decomposition_results]
assert len(decomposition_ids) == len(set(decomposition_ids)) and set(decomposition_ids) == {doc.metadata['source_id'] for doc in all_results}, 'decomposition_results는 all_results를 unique_documents로 정리한 목록입니다.'
assert [doc.metadata['source_id'] for doc in decomposition_before] == [doc.metadata['source_id'] for doc in hybrid.invoke(decomposition_question)][:TOP_K], 'decomposition_before에는 hybrid로 원질문을 검색한 결과의 앞 TOP_K개를 담으세요.'
print("✅ 6번 통과!")


아래 `[제공 코드]`는 교안 02의 답변 문맥 함수 `format_context`와 답변 체인 `answer_chain`입니다. 답변은 `[원문 ID]`를 인용합니다.


In [ ]:
# [제공 코드]
# 본문과 조건 메타데이터를 함께 줘 연도·분류·쪽수도 답변에서 확인할 수 있게 합니다.
def format_context(documents):
    """실제 검색 문서의 본문·메타데이터·출처를 답변 문맥으로 만듭니다."""
    return "\n\n".join(
        f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n"
        f"메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for doc in documents
    )


In [ ]:
# [제공 코드]
# 실제 검색 원문에 있는 내용으로만 원질문에 답합니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 본문과 메타데이터에서 직접 확인되는 내용만 한글로 답하세요. "
               "질문에 직접 답하는 내용만 최대 3개 항목으로 쓰고, 항목마다 1~2문장과 [원문 ID] 인용을 넣으세요. 관련 없는 문서는 언급하지 마세요. "
               "원문에 없는 절차·조건·조언을 추가하지 마세요. "
               "원문의 권고나 가능성을 의무로 바꾸지 말고, 수치·단위·비율·조건을 그대로 보존하세요. 사용자가 제시한 방안을 원문이 허용하거나 정당화한다고 추론하지 마세요. "
               "근거가 부족한 부분은 확인할 수 없다고 말하고, 추가 결론은 쓰지 마세요."),
    ("human", "질문: {question}\n\n원문 근거:\n{context}"),
])
answer_chain = answer_prompt | llm | StrOutputParser()


## 7. 실제 검색한 원문으로 답하고 기록을 저장합니다

**배경**: 질의 변환의 목적은 원질문에 근거 있는 답을 하는 것입니다. 생성한 하위 질문은 검색에만 쓰고, 답변은 원질문과 실제 검색한 원문으로 만듭니다.

**요구사항**:

- **`answer`**: `answer_chain`에 `{"question": decomposition_question, "context": format_context(decomposition_results)}`를 넣은 답변 문자열입니다. 출력하세요. 하위 질문이 아니라 6번의 원질문으로 답합니다.
- **`report`**: 키가 `question`, `subquestions`, `source_ids`, `answer`인 딕셔너리입니다. `question`은 `decomposition_question`, `subquestions`는 `subquestions.questions`, `source_ids`는 `decomposition_results`의 원문 ID 목록(순서 유지), `answer`는 위 답변입니다.
- **저장**: `save_json`으로 `report`를 `release_search_report.json`에 저장하세요. 파일 이름만 넘깁니다(`output/` 폴더는 함수가 붙입니다).

**예시**: 답변에는 `[release4]`처럼 원문 ID 인용이 붙습니다. 인용한 원문을 다시 읽어 답변이 원문에 있는 내용만 말하는지 확인하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 원질문과 실제 Document만으로 답변 입력을 만들고, 검색 과정과 답변을 한 딕셔너리로 저장합니다.

세부구현:
1. format_context로 decomposition_results를 문맥 문자열로 만들어 answer_chain에 넣습니다.
2. 원질문·하위 질문·원문 ID·답변으로 report를 만듭니다.
3. save_json으로 저장합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert isinstance(answer, str) and answer.strip(), 'answer에는 answer_chain이 만든 답변 문자열을 담으세요.'
assert (output_dir / 'release_search_report.json').exists(), 'save_json으로 report를 release_search_report.json에 저장하세요. 파일 이름에 output/을 붙이지 마세요.'
assert isinstance(report, dict) and set(report) == {'question', 'subquestions', 'source_ids', 'answer'}, 'report의 키는 question, subquestions, source_ids, answer 네 개입니다.'
assert report['question'] == decomposition_question, 'report의 question에는 하위 질문이 아니라 6번의 원질문을 넣으세요.'
assert report['subquestions'] == subquestions.questions, 'report의 subquestions에는 subquestions.questions를 넣으세요.'
assert report['source_ids'] == [doc.metadata['source_id'] for doc in decomposition_results], 'report의 source_ids에는 decomposition_results의 원문 ID를 순서대로 넣으세요.'
assert report['answer'] == answer, 'report의 answer에는 방금 만든 answer를 넣으세요.'
assert json.loads((output_dir / 'release_search_report.json').read_text(encoding='utf-8')) == report, '현재 report를 release_search_report.json에 다시 저장하세요.'
print("✅ 7번 통과!")


## 8. 상황에 맞는 질의 변환을 고르고 결과를 해석합니다

**배경**: 변환마다 LLM 호출과 검색이 늘어나므로 질문의 성격에 맞는 방법만 골라 써야 합니다. 교안 02 각 절의 ‘유용한 경우’와 정리표를 떠올리며 판단합니다.

**요구사항**:

- **상황별 선택**: 아래 세 요청에 가장 알맞은 변환을 하나씩 고르고 그 이유를 쓰세요.
  - (가) ‘2.3 계열의 stable 릴리스에서 바뀐 클러스터 기능’
  - (나) ‘DB가 표들 사이의 관계를 다루는 방식’처럼 문서와 다른 일상 표현으로 물을 때
  - (다) ‘인덱스 변경과 드라이버 변경을 각각 알려 줘’
- **가상문단과 근거**: HyDE가 ‘모든 인덱스는 자동으로 생성된다’라는 가상문장을 만들었습니다. 이 문장을 검색 입력으로 써도 되는지, 답변에 인용해도 되는지 구분해 쓰세요.
- **결과 해석**: 3~6번 가운데 Recall이 오르지 않았거나 떨어진 변환 하나를 골라, 여러분의 출력(가상문단·생성 질문·배경 질문·결과 원문 ID)을 근거로 이유를 쓰세요. 변환 전부터 1.0이었던 문항은 빼고, 모두 올랐다면 가장 적게 오른 변환을 고릅니다.

**예시**: 항목마다 한두 문장씩 씁니다. 결과 해석은 실제 출력 값을 인용하세요.

<details><summary>힌트</summary>

```text
접근방법:
- 각 요청이 조건, 표현 차이, 여러 요구 가운데 무엇에 해당하는지부터 가립니다.

세부구현:
1. 메타데이터 조건이 있는지, 표현이 문서와 다른지, 요구가 둘 이상인지 확인합니다.
2. 조건은 Self-Query, 표현 차이는 Multi-Query나 HyDE, 여러 요구는 Decomposition과 연결합니다.
3. 가상문단은 검색 입력과 답변 근거를 나누어 판단합니다.
4. 결과 해석은 변환 전후 결과 ID와 생성된 문장을 나란히 놓고 원인을 찾습니다.
```

</details>


*(여기에 판단과 근거를 서술하세요)*
